
<div style="
    width: 100%;
    max-width: 100%;
    box-sizing: border-box;
    overflow-wrap: anywhere;
    border: 1px solid #c7d2fe;
    border-left: 6px solid #6366f1;
    background: linear-gradient(135deg, #f8faff 0%, #eef2ff 55%, #f5f3ff 100%);
    border-radius: 12px;
    padding: 20px;
    color: #1e293b;
    font-family: system-ui, -apple-system, 'Segoe UI', Roboto, sans-serif;
    line-height: 1.8;
">

<h2 style="color: #3730a3; font-size: 24px; font-weight: 750; margin: 0 0 8px 0;">
    The Reflection Pattern
</h2>

<p style="color: #4f46e5; font-size: 15px; font-weight: 600; margin: 0 0 18px 0;">
    Generating and Refining a Persian LinkedIn Post
</p>

<p style="margin: 0 0 20px 0;">
    This notebook implements the <strong>Reflection Pattern</strong>,
    following the essay workflow approach to generate, critique, and
    refine Persian-language posts for LinkedIn.
</p>

<div style="box-sizing: border-box; max-width: 100%; background: #ffffff; border: 1px solid #dbeafe; border-radius: 9px; padding: 14px 16px; margin-bottom: 14px; overflow-wrap: anywhere;">
    <p style="margin: 0 0 8px 0; color: #4338ca; font-weight: 700; font-size: 15px;">
        Part 1 — Basic Reflection
    </p>
    <p style="margin: 0;">
        Implements
        <code style="background: #e0e7ff; color: #312e81; padding: 3px 6px; border-radius: 5px; font-weight: 700;">generate_draft</code>,
        <code style="background: #e0e7ff; color: #312e81; padding: 3px 6px; border-radius: 5px; font-weight: 700;">reflect_on_draft</code>,
        and
        <code style="background: #e0e7ff; color: #312e81; padding: 3px 6px; border-radius: 5px; font-weight: 700;">revise_draft</code>
        with two measurable constraints: a maximum of
        <strong>300 characters</strong> and <strong>3–5 hashtags</strong>.
        The complete workflow is tested on two different topics.
    </p>
</div>

<div style="box-sizing: border-box; max-width: 100%; background: #ffffff; border: 1px solid #ddd6fe; border-radius: 9px; padding: 14px 16px; margin-bottom: 18px; overflow-wrap: anywhere;">
    <p style="margin: 0 0 8px 0; color: #6d28d9; font-weight: 700; font-size: 15px;">
        Part 2 — External Feedback
    </p>
    <p style="margin: 0;">
        Introduces
        <code style="background: #f3e8ff; color: #6b21a8; padding: 3px 6px; border-radius: 5px; font-weight: 700;">check_post_quality</code>,
        a rule-based Python function that measures post length and
        hashtag count without using an LLM. These objective quality
        signals are passed into the reflection prompt, following the
        same principle as
        <code style="background: #f3e8ff; color: #6b21a8; padding: 3px 6px; border-radius: 5px; font-weight: 700;">refine_sql_external_feedback</code>.
    </p>
</div>

<div style="box-sizing: border-box; max-width: 100%; border-top: 1px solid #c7d2fe; padding-top: 12px; color: #475569; font-size: 13px; overflow-wrap: anywhere;">
    <strong style="color: #4338ca;">Display note:</strong>
    Outputs are presented as styled, right-to-left (RTL) HTML cards
    using helper functions defined in
    <code style="background: #e2e8f0; color: #1e293b; padding: 2px 5px; border-radius: 4px; font-weight: 600;">utils.py</code>,
    making Persian text easier to read.
</div>

</div>


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from utils import render_results_html, render_comparison_html, render_finals_html

# Load .env file
load_dotenv()

# Read values
api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("OPENAI_BASE_URL")

# Create client
client = OpenAI(
    api_key=api_key,
    base_url=base_url
)


DEFAULT_MODEL = "gpt-4o-mini"  # Default model to use for generating responses

## Part 1 — The Three Core Functions

In [2]:
def generate_draft(topic, tone="حرفه‌ای", model=DEFAULT_MODEL):
    """Generates a first draft of a LinkedIn post in Persian (Farsi)."""
    prompt = f"""یک پست برای شبکه‌ی اجتماعی لینکدین درباره‌ی موضوع «{topic}» بنویس.

لحن پست: {tone}

محدودیت‌های الزامی:
- حداکثر ۳۰۰ کاراکتر (برای کل پست)
- بین ۳ تا ۵ هشتگ مرتبط در انتهای پست

فقط متن پست را بنویس، بدون هیچ توضیح اضافه و بدون علامت نقل‌قول.
متن پست باید کاملاً فارسی باشد (هشتگ‌ها می‌توانند فارسی یا لاتین باشند)."""

    response = client.chat.completions.create(
        model=model,
        temperature=1.0,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content.strip()


In [3]:
def reflect_on_draft(draft, quality_signals=None, model=DEFAULT_MODEL):
    """Critiques a LinkedIn post draft.

    If quality_signals is None → behaves like the original Part-1 version
    (general, signal-free critique). If a quality_signals dict is given, the
    exact measured numbers and violated limits are injected into the prompt,
    mirroring the refine_sql_external_feedback pattern from 02_02.
    """
    if quality_signals is None:
        # ---- Part 1: general critique without any external signals ----
        prompt = f"""زیر یک پیش‌نویس از یک پست لینکدین آمده است:

---
{draft}
---

این پست را از این دیدگاه‌ها نقد کن:
- آیا طول پست برای یک پست خوب لینکدین مناسب است؟
- آیا تعداد و کیفیت هشتگ‌ها مناسب است؟
- آیا لحن پست جذاب و مناسب مخاطب حرفه‌ای است؟

نقد را کوتاه و در چند جمله بیان کن. نقد باید فارسی باشد."""
    else:
        # ---- Part 2: critique with concrete external quality signals ----
        prompt = f"""زیر یک پیش‌نویس از یک پست لینکدین آمده است:

---
{draft}
---

نتایج بررسی خودکار کیفیت این پست:
- طول پست: {quality_signals['length']} کاراکتر (آیا در محدودیت مجاز است؟ {quality_signals['within_length_limit']})
- تعداد هشتگ‌ها: {quality_signals['hashtag_count']} (آیا در محدوده‌ی مجاز است؟ {quality_signals['within_hashtag_range']})

با استفاده از همین اعداد دقیق، ایرادات پست را به‌صراحت و با عدد مشخص بیان کن.
محدودیت‌ها عبارت‌اند از: حداکثر ۳۰۰ کاراکتر، و بین ۳ تا ۵ هشتگ.
اگر هر محدودیتی نقض شده، دقیقاً بگو چقدر نقض شده و چرا مشکل‌ساز است.
نقد باید فارسی باشد."""

    response = client.chat.completions.create(
        model=model,
        temperature=1.0,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content.strip()


In [4]:
def revise_draft(draft, feedback, model=DEFAULT_MODEL):
    """Revises the post based on the feedback received (Farsi output)."""
    prompt = f"""پیش‌نویس اولیه‌ی پست لینکدین:
---
{draft}
---

بازخورد دریافت‌شده درباره این پیش‌نویس:
---
{feedback}
---

بر اساس این بازخورد، یک نسخه‌ی بازنویسی‌شده از پست بنویس.
محدودیت‌های حداکثر ۳۰۰ کاراکتر و بین ۳ تا ۵ هشتگ باید رعایت شوند.
فقط متن نهایی پست را بنویس و متن باید فارسی باشد."""

    response = client.chat.completions.create(
        model=model,
        temperature=1.0,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content.strip()


## Combined Function to Run the Full Workflow

In [5]:
def run_linkedin_workflow(topic, tone="حرفه‌ای", model="gpt-4o-mini"):
    """Runs all three steps (generate, reflect, revise) in sequence."""
    draft = generate_draft(topic, tone=tone, model=model)
    feedback = reflect_on_draft(draft, model=model)
    revised = revise_draft(draft, feedback, model=model)

    return {
        "topic": topic,
        "tone": tone,
        "draft": draft,
        "feedback": feedback,
        "revised": revised,
    }


## Test with Two Different Topics

In [35]:
# Two different topics (Farsi), as required by the task:
#   1) introducing a new training course
#   2) announcing a work achievement
result_1 = run_linkedin_workflow("معرفی یک دوره‌ی آموزشی جدید در زمینه‌ی هوش مصنوعی")
result_2 = run_linkedin_workflow("اعلام یک دستاورد کاری: تکمیل موفق یک پروژه‌ی بزرگ")

results = [result_1, result_2]

render_results_html(results)


<div style="
    width: 100%;
    max-width: 100%;
    box-sizing: border-box;
    overflow-wrap: anywhere;
    border: 1px solid #bfdbfe;
    border-left: 6px solid #2563eb;
    background: linear-gradient(135deg, #eff6ff 0%, #f8faff 100%);
    border-radius: 12px;
    padding: 20px;
    color: #1e293b;
    font-family: system-ui, -apple-system, 'Segoe UI', Roboto, sans-serif;
    line-height: 1.8;
">

<h3 style="
    color: #1d4ed8;
    font-size: 20px;
    margin: 0 0 12px 0;
">
    Discussion Question — Temperature Settings
</h3>

<p style="margin: 0 0 14px 0;">
    Why does the essay workflow use
    <code style="background: #dbeafe; color: #1e40af; padding: 2px 6px; border-radius: 5px; font-weight: 700;">temperature=1.0</code>,
    while SQL generation uses
    <code style="background: #dbeafe; color: #1e40af; padding: 2px 6px; border-radius: 5px; font-weight: 700;">temperature=0</code>?
</p>

<div style="
    background: #ffffff;
    border: 1px solid #dbeafe;
    border-radius: 8px;
    padding: 14px 16px;
    margin-bottom: 12px;
">
    <p style="margin: 0 0 6px 0; color: #1d4ed8; font-weight: 700;">
        SQL Generation — Temperature 0
    </p>
    SQL needs accurate, predictable results. A lower temperature reduces randomness and makes outputs more consistent.
</div>

<div style="
    background: #ffffff;
    border: 1px solid #bfdbfe;
    border-radius: 8px;
    padding: 14px 16px;
    margin-bottom: 14px;
">
    <p style="margin: 0 0 6px 0; color: #1d4ed8; font-weight: 700;">
        Text Generation and Reflection — Temperature 1.0
    </p>
    Writing allows many good answers. A higher temperature encourages variety in ideas, wording, and structure, helping the model explore different drafts and critiques.
</div>

<div style="
    background: #dbeafe;
    border-radius: 8px;
    padding: 12px 15px;
    color: #1e3a8a;
">
    <strong>Key takeaway:</strong>
    Use lower temperatures for predictable tasks and higher temperatures for creative tasks. Higher temperature encourages variety but does not guarantee better results.
</div>

</div>


---
## Part 2 — Adding External Feedback

In [7]:
def check_post_quality(post_text, max_length=300, min_hashtags=3, max_hashtags=5):
    """Checks post quality using simple Python rules (no LLM call).

    Returns a dict with both the raw measured values and the boolean
    pass/fail against the constraints defined in Part 1:
        length, within_length_limit, hashtag_count, within_hashtag_range.

    Note: len() counts Python characters (code points). For Persian text this
    is the right notion for a character-count constraint, even though Persian
    glyphs may render wider visually.
    """
    length = len(post_text)
    within_length_limit = length <= max_length

    hashtags = [word for word in post_text.split() if word.startswith("#")]
    hashtag_count = len(hashtags)
    within_hashtag_range = min_hashtags <= hashtag_count <= max_hashtags

    return {
        "length": length,
        "within_length_limit": within_length_limit,
        "hashtag_count": hashtag_count,
        "within_hashtag_range": within_hashtag_range,
    }


In [8]:
# Quick test of the function on a manual Persian sample
sample_check = check_post_quality("این یک پست نمونه است که هیچ هشتگی ندارد.")
sample_check


{'length': 40,
 'within_length_limit': True,
 'hashtag_count': 0,
 'within_hashtag_range': False}

## New Reflect Version That Sees the Quality Signals

In [31]:
# The single reflect_on_draft() function now handles BOTH modes:
#   - reflect_on_draft(draft)                  -> Part 1: general critique (no signals)
#   - reflect_on_draft(draft, quality_signals) -> Part 2: critique with exact measured numbers
# This mirrors refine_sql_external_feedback from 02_02, where the *actual execution result*
# of the SQL (not just the query text) is fed back into the prompt.
#
# We keep the no-signal behaviour available (quality_signals=None default) so the two
# versions can be compared on the SAME post, as the task requires.

## Comparison: Feedback With Signals vs. Without Signals

In [ ]:
# An intentionally flawed Persian post: too long (425 > 300) and only 1 hashtag (< 3).
# This violates BOTH constraints, so the signals-aware critique can name exact numbers.
bad_post = (
"""امروز می‌خواهم یکی از مهم‌ترین تجربه‌های کاری خودم را با شما به اشتراک بگذارم. 
در چند ماه گذشته، همراه با تیمم روی پروژه‌ای کار کردیم که چالش‌های فنی زیادی داشت و برای رسیدن به نتیجه مجبور شدیم بارها راهکارهای مختلف را آزمایش کنیم.
در نهایت توانستیم پروژه را با موفقیت به پایان برسانیم و تجربه‌های ارزشمندی درباره حل مسئله، همکاری تیمی، مدیریت زمان و تصمیم‌گیری به دست آوریم. 
این مسیر به من یاد داد که موفقیت فقط به نتیجه نهایی وابسته نیست، بلکه به یادگیری، پشتکار و همکاری افراد نیز بستگی دارد.
از تمام هم‌تیمی‌هایم که در این مسیر کنارم بودند تشکر می‌کنم و امیدوارم در پروژه‌های آینده بتوانیم نتایج بهتر و ارزشمندتری به دست آوریم."""
)

quality_signals = check_post_quality(bad_post)
print("Quality signals:", quality_signals)

feedback_without_signals = reflect_on_draft(bad_post)                              # Part 1 mode
feedback_with_signals    = reflect_on_draft(bad_post, quality_signals=quality_signals)  # Part 2 mode

render_comparison_html(quality_signals, feedback_without_signals, feedback_with_signals)

Quality signals: {'length': 633, 'within_length_limit': False, 'hashtag_count': 0, 'within_hashtag_range': False}


## Final Answers: Revised Post From Each Feedback

Now that we have **two feedbacks** for the same `bad_post` (with and without signals),
we feed each one back into `revise_draft()` — reusing the existing function, no new
function needed — to produce **two final posts** and compare them. The signal-aware
feedback should yield a final post that fixes the *exact* violations (length & hashtags).

In [33]:
# Reuse the existing revise_draft() — NO new function needed.
# We just call it twice: once with the signal-free feedback, once with the signal-aware feedback.
final_without_signals = revise_draft(bad_post, feedback_without_signals)   # final from Part-1 style feedback
final_with_signals    = revise_draft(bad_post, feedback_with_signals)      # final from Part-2 style feedback

# Sanity-check both finals against the same rules, so we can SEE which one actually satisfies the limits.
check_final_no   = check_post_quality(final_without_signals)
check_final_yes  = check_post_quality(final_with_signals)

print("Final WITHOUT signals ->", check_final_no)
print("Final WITH    signals ->", check_final_yes)

render_finals_html(final_without_signals, check_final_no,
                   final_with_signals,    check_final_yes)

Final WITHOUT signals -> {'length': 304, 'within_length_limit': False, 'hashtag_count': 5, 'within_hashtag_range': True}
Final WITH    signals -> {'length': 300, 'within_length_limit': True, 'hashtag_count': 5, 'within_hashtag_range': True}
